<a href="https://www.kaggle.com/code/ameythakur20/birdclef-2026-perch-v2-bayesian-fusion" target="_blank">
    <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle">
</a>

# BirdCLEF+ 2026: Perch v2 Embedding Probes with Bayesian Spatial-Temporal Fusion

Main inference notebook for the BirdCLEF 2026 competition. This pipeline uses Perch v2 and Logistic Regression probes with Bayesian fusion for score refinement.

### Resources:

*   **Competition**: [BirdCLEF+ 2026](https://www.kaggle.com/competitions/birdclef-2026)
*   **Environment Setup**: [birdclef-2026-tensorflow-2-20-setup](https://www.kaggle.com/code/ameythakur20/birdclef-2026-tensorflow-2-20-setup)

Author: [Amey Thakur](https://www.kaggle.com/ameythakur20)


## 1. Install Dependencies

Perch v2 SavedModel uses StableHLO operations introduced in TF 2.20. Kaggle default is 2.19, so pre-staged wheels are installed from an attached dataset to avoid internet dependency during submission.


In [ ]:
import subprocess, sys, os
from pathlib import Path

# Robust wheel discovery for BirdCLEF 2026 TF 2.20 Setup
_WHL = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "birdclef-2026-tensorflow-2-20-setup" in root and "wheel" in root:
        _WHL = Path(root)
        break

if not _WHL:
    candidates = [
        Path("/kaggle/input/notebooks/ameythakur20/birdclef-2026-tensorflow-2-20-setup/wheel"),
        Path("/kaggle/input/birdclef-2026-tensorflow-2-20-setup/wheel"),
        Path("/kaggle/input/ameythakur20/birdclef-2026-tensorflow-2-20-setup/wheel")
    ]
    _WHL = next((p for p in candidates if p.exists()), None)

if _WHL and _WHL.exists():
    print(f"Directory found: {_WHL}")
    tb_whl = next(_WHL.glob("tensorboard-2.20.0*.whl"), None)
    tf_whl = next(_WHL.glob("tensorflow-2.20.0*.whl"), None)
    
    if tb_whl and tf_whl:
        # Quietly force reinstall of the specific 2.20 wheels to clear any pre-installed conflicts
        # NOTE: Restart the session AFTER running this if you get NotFoundError/UndefinedSymbol on import
        print(f"Installing {tf_whl.name} environment...")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", str(tb_whl)], check=True, capture_output=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", str(tf_whl)], check=True, capture_output=True)
            
            import importlib
            importlib.invalidate_caches()
            print("Success. (Restart Session recommended if re-importing TensorFlow)")
        except Exception as e:
            print(f"Installation failed: {e}")
    else:
        print(f"Error: Missing wheels in {_WHL}")
else:
    print("Wheel directory not found.")


## 2. Imports and Runtime Mode

CPU-only execution is enforced via CUDA_VISIBLE_DEVICES to comply with the competition constraint. All random seeds are pinned for reproducibility.


In [ ]:
import os, sys, warnings, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import gc, random, re
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tf.get_logger().setLevel(logging.ERROR)
tf.experimental.numpy.experimental_enable_numpy_behavior()


### Configuration

All paths and hyperparameters are centralized. MODE controls whether expensive grid searches run (train) or frozen parameters are used (submit). Cache auto-detection searches candidate directories for pre-computed Perch outputs.


In [ ]:
import sys
import os
TOOLBOX_PATH = "/kaggle/usr/lib/notebooks/ameythakur20/kaggle_toolbox"
if TOOLBOX_PATH not in sys.path:
    sys.path.append(TOOLBOX_PATH)

from kaggle_toolbox import seed_everything
seed_everything(42)  # Maintain reproducibility tracking


## 3. Exploratory Data Analysis

Four metadata files are loaded: taxonomy (234 species), train recordings, soundscape annotations, and sample submission. Understanding taxonomic and recording distributions informs the fusion and smoothing strategy.


In [ ]:
from pathlib import Path
import numpy as np
import os
import sys

def struct(**kwargs): return type("struct", (), kwargs)()

CFG = struct(
    MODE           = "submit",
    BASE           = Path("/kaggle/input/competitions/birdclef-2026"),
    META_DIR       = Path("/kaggle/input/datasets/jaejohn/perch-meta"),
    MODEL_DIR      = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"),
    WORK_CACHE     = Path("/kaggle/working/cache"),
    CACHE_DIR      = None,
    DRYRUN_FILES   = 20,
    N_WINDOWS      = 12,
    SAMPLE_RATE    = 32_000,
    DURATION       = 60,
    WINDOW         = 5,
    WINDOW_SAMPLES = 5 * 32_000,
    FILE_SAMPLES   = 60 * 32_000,
    BATCH_FILES    = 10,
    N_SPLITS       = 5,
    PCA_DIM        = 64,
    MIN_POS        = 3,
    PROBE_ALPHA    = 0.3,
    LAMBDA_EVENT   = 0.6,
    LAMBDA_TEXTURE = 0.4,
    LAMBDA_PROXY_TEXTURE = 0.3,
    SMOOTH_EVENT   = np.array([0.05, 0.15, 0.60, 0.15, 0.05], dtype=np.float32),
    SMOOTH_TEXTURE = np.array([0.10, 0.20, 0.40, 0.20, 0.10], dtype=np.float32),
    POST_POWER     = 1.66,
)

CFG.WORK_CACHE.mkdir(parents=True, exist_ok=True)
print(f"Mode: {CFG.MODE} initialized.")


In [ ]:
taxonomy        = pd.read_csv(CFG.BASE / "taxonomy.csv")
train_meta      = pd.read_csv(CFG.BASE / "train.csv")
soundscape_raw  = pd.read_csv(CFG.BASE / "train_soundscapes_labels.csv")
sample_sub      = pd.read_csv(CFG.BASE / "sample_submission.csv")

# deduplicate soundscape annotations
soundscape_lbls = soundscape_raw.drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

print(f"Taxonomy species    : {len(taxonomy)}")
print(f"Train recordings    : {len(train_meta):,}")
print(f"Soundscape rows     : {len(soundscape_lbls):,} unique "
      f"(dropped {len(soundscape_raw) - len(soundscape_lbls):,} duplicates)")
print(f"Submission columns  : {N_CLASSES}")


In [ ]:
# taxonomic class distribution -- drives the texture vs event split
class_counts = taxonomy["class_name"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6"]
class_counts.plot.barh(ax=ax, color=colors[:len(class_counts)])
ax.set_xlabel("Number of Species")
ax.set_title("Species Count by Taxonomic Class")
for i, v in enumerate(class_counts.values):
    ax.text(v + 0.5, i, str(v), va="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# recording distribution per species
recs_per_species = train_meta["primary_label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
recs_per_species.head(20).sort_values().plot.barh(ax=axes[0], color="#3498db")
axes[0].set_title("Top 20 Species by Recording Count")
axes[0].set_xlabel("Recordings")
recs_per_species.tail(20).sort_values().plot.barh(ax=axes[1], color="#e74c3c")
axes[1].set_title("Bottom 20 Species by Recording Count")
axes[1].set_xlabel("Recordings")
plt.tight_layout()
plt.show()

print(f"Total species with recordings: {len(recs_per_species)}")
print(f"Median recordings per species: {recs_per_species.median():.0f}")


In [ ]:
# recording source and quality
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
coll_counts = train_meta["collection"].value_counts()
coll_counts.plot.pie(ax=axes[0], autopct="%1.1f%%", colors=["#3498db", "#2ecc71"], startangle=90)
axes[0].set_title("Recording Source")
axes[0].set_ylabel("")

xc_ratings = train_meta[train_meta["collection"] == "XC"]["rating"]
xc_ratings[xc_ratings > 0].hist(ax=axes[1], bins=10, color="#3498db", edgecolor="white")
axes[1].set_title("Xeno-canto Recording Quality Ratings")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# geographic spread of training recordings
geo = train_meta.dropna(subset=["latitude", "longitude"])
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(geo["longitude"], geo["latitude"], c="#3498db", alpha=0.15, s=5, edgecolors="none")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Training Recording Locations ({len(geo):,} recordings)")
plt.tight_layout()
plt.show()


## 4. Label Preprocessing

Soundscape labels contain semicolon-separated species codes per 5-second window. Labels are aggregated per window, filename metadata (site, date, hour) is parsed, and a multi-hot truth matrix is built for OOF validation.


In [ ]:
FNAME_RE = re.compile(
    r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg"
)

def parse_labels(x):
    if pd.isna(x): return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_labels(x)))

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": None, "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

# aggregate labels per 5-second window
sc_clean = (
    soundscape_lbls.groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels).reset_index(name="label_list")
)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"]  = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)
meta_cols = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta_cols], axis=1)

# identify fully-labeled files (all 12 windows annotated)
wpf = sc_clean.groupby("filename").size()
full_files = sorted(wpf[wpf == CFG.N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

# multi-hot label matrix
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    for lbl in labels:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

full_truth = sc_clean[sc_clean["file_fully_labeled"]].sort_values(["filename", "end_sec"]).reset_index(drop=True)
Y_FULL_TRUTH = Y_SC[sc_clean[sc_clean["file_fully_labeled"]].index.to_numpy()]

print(f"Fully-labeled files : {len(full_files)}")
print(f"Trusted windows     : {len(full_truth)}")
print(f"Active classes      : {int((Y_FULL_TRUTH.sum(axis=0) > 0).sum())}")


In [ ]:
# species activity in labeled soundscapes
species_activity = Y_FULL_TRUTH.sum(axis=0)
active_mask = species_activity > 0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
active_counts = species_activity[active_mask]
axes[0].hist(active_counts, bins=30, color="#2ecc71", edgecolor="white")
axes[0].set_title(f"Positive Windows per Species (n={len(active_counts)})")
axes[0].set_xlabel("Positive Windows")
axes[0].set_ylabel("Number of Species")
axes[0].axvline(x=CFG.MIN_POS, color="red", linestyle="--", label=f"MIN_POS={CFG.MIN_POS}")
axes[0].legend()

axes[1].pie(
    [active_mask.sum(), (~active_mask).sum()],
    labels=[f"Active ({active_mask.sum()})", f"Inactive ({(~active_mask).sum()})"],
    colors=["#2ecc71", "#e74c3c"], autopct="%1.0f%%", startangle=90,
)
axes[1].set_title("Species Presence in Labeled Soundscapes")
plt.tight_layout()
plt.show()


In [ ]:
# site x hour occurrence heatmap
site_hour = sc_clean.groupby(["site", "hour_utc"]).size().reset_index(name="windows")
pivot = site_hour.pivot(index="site", columns="hour_utc", values="windows").fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Labeled Windows by Site and Hour (UTC)")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Recording Site")
plt.tight_layout()
plt.show()


## 5. Perch v2 Model and Species Mapping

Perch v2 was pretrained on 14,795 species identified by scientific name. Each competition species is mapped to its Perch class index. Species with no direct match are left unmapped or, for unmapped amphibians, assigned a genus-level proxy using the max logit across congener indices.


In [ ]:
import subprocess, sys

birdclassifier = tf.saved_model.load(str(CFG.MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]
print("Perch v2 loaded.")

bc_labels = (
    pd.read_csv(CFG.MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_ = taxonomy.copy()
taxonomy_["scientific_name"] = taxonomy_["scientific_name"].astype(str)
mapping = taxonomy_.merge(bc_labels[["scientific_name", "bc_index"]], on="scientific_name", how="left")
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK       = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS        = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS      = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped   : {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Unmapped : {(~MAPPED_MASK).sum()}")


In [ ]:
# texture vs event split: frogs and insects produce continuous choruses,
# birds produce transient calls -- this drives different fusion weights
CLASS_NAME_MAP = taxonomy_.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA], dtype=np.int32)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA], dtype=np.int32)

idx_mapped_active_texture  = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event    = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event   = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32)

# genus-level proxy for unmapped amphibians: if a species is absent from Perch
# but a congener exists, the max logit across congener indices serves as proxy
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)].copy()

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[bc_labels["scientific_name"].str.match(rf"^{re.escape(genus)}\s", na=False)]
    if len(hits) > 0:
        proxy_map[str(row["primary_label"])] = hits["bc_index"].astype(int).tolist()

SELECTED_PROXY_TARGETS = sorted([t for t in proxy_map if CLASS_NAME_MAP.get(t) == "Amphibia"])
selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32) for t in SELECTED_PROXY_TARGETS
}
idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Active texture classes : {len(idx_active_texture)}")
print(f"Active event classes   : {len(idx_active_event)}")
print(f"Frog proxy targets     : {SELECTED_PROXY_TARGETS}")


## 6. Perch Inference Engine

Each 60-second soundscape is split into 12 non-overlapping 5-second windows. Perch returns raw logits over its vocabulary and a 1536-d embedding per window. Mapped species logits are extracted directly; genus-proxy scores use the max logit across congener indices.


In [ ]:
def read_soundscape_60s(path):
    """Load a 60-second soundscape, padding or truncating to exact length."""
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < CFG.FILE_SAMPLES:
        y = np.pad(y, (0, CFG.FILE_SAMPLES - len(y)))
    return y[:CFG.FILE_SAMPLES]


def infer_perch_batch(paths, verbose=True):
    """Run Perch v2 on a list of soundscape paths.
    Returns metadata DataFrame, raw score matrix, and embedding matrix."""
    paths   = [Path(p) for p in paths]
    n_rows  = len(paths) * CFG.N_WINDOWS
    row_ids    = np.empty(n_rows, dtype=object)
    filenames  = np.empty(n_rows, dtype=object)
    sites      = np.empty(n_rows, dtype=object)
    hours      = np.empty(n_rows, dtype=np.int16)
    scores     = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536),      dtype=np.float32)
    write_row = 0
    itr = tqdm(range(0, len(paths), CFG.BATCH_FILES), desc="Perch", disable=not verbose)
    for start in itr:
        batch  = paths[start : start + CFG.BATCH_FILES]
        x      = np.empty((len(batch) * CFG.N_WINDOWS, CFG.WINDOW_SAMPLES), dtype=np.float32)
        bstart = write_row
        for bi, path in enumerate(batch):
            x[bi * CFG.N_WINDOWS : (bi + 1) * CFG.N_WINDOWS] = read_soundscape_60s(path).reshape(
                CFG.N_WINDOWS, CFG.WINDOW_SAMPLES)
            meta = parse_soundscape_filename(path.name)
            row_ids[write_row : write_row + CFG.N_WINDOWS] = [f"{path.stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row : write_row + CFG.N_WINDOWS] = path.name
            sites[write_row : write_row + CFG.N_WINDOWS]     = meta["site"]
            hours[write_row : write_row + CFG.N_WINDOWS]     = meta["hour_utc"]
            write_row += CFG.N_WINDOWS
        out    = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = out["label"].numpy().astype(np.float32)
        emb    = out["embedding"].numpy().astype(np.float32)
        scores[bstart:write_row, MAPPED_POS] = logits[: write_row - bstart, MAPPED_BC_INDICES]
        embeddings[bstart:write_row] = emb
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row, pos] = logits[: write_row - bstart, bc_idx_arr].max(axis=1)
        del x, out, logits, emb
        gc.collect()
    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours})
    return meta_df, scores, embeddings


In [ ]:
# load cache or run inference on labeled soundscapes
if CFG.CACHE_DIR is not None:
    print(f"Loading Perch cache from: {CFG.CACHE_DIR}")
    meta_full       = pd.read_parquet(CFG.CACHE_DIR / "full_perch_meta.parquet")
    arr             = np.load(CFG.CACHE_DIR / "full_perch_arrays.npz")
    scores_full_raw = arr["scores_full_raw"].astype(np.float32)
    emb_full        = arr["emb_full"].astype(np.float32)
else:
    print("No cache found. Running Perch on fully-labeled soundscapes...")
    full_paths = [CFG.BASE / "train_soundscapes" / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_batch(full_paths)
    meta_full.to_parquet(CFG.WORK_CACHE / "full_perch_meta.parquet", index=False)
    np.savez_compressed(
        CFG.WORK_CACHE / "full_perch_arrays.npz",
        scores_full_raw=scores_full_raw, emb_full=emb_full,
    )
    print(f"Cache saved to {CFG.WORK_CACHE}")

# align ground truth to inference row order using row_id identity
Y_SC_DF = pd.DataFrame(Y_SC, index=sc_clean["row_id"])
Y_FULL = Y_SC_DF.loc[meta_full["row_id"]].to_numpy().astype(np.uint8)

print(f"scores_full_raw : {scores_full_raw.shape}  {scores_full_raw.dtype}")
print(f"emb_full        : {emb_full.shape}  {emb_full.dtype}")
print(f"Y_FULL          : {Y_FULL.shape}")


## 7. Bayesian Prior Fusion

Raw Perch logits reflect global species probabilities but ignore deployment context. Prior tables capture site, hour, and joint site-hour prevalence, each shrunk toward the global mean using sample-size-dependent mixing. Texture species get higher lambda (1.0) because their presence correlates strongly with location and time; birds get lower lambda (0.4) because the Perch logit already carries strong signal.


In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique())
    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique())

    site_to_i, site_n, site_p = {}, [], []
    for s in site_keys:
        mask = prior_df['site'].astype(str).values == s
        site_to_i[s] = len(site_n)
        site_n.append(mask.sum())
        site_p.append(Y_prior[mask].mean(axis=0))
    site_n = np.array(site_n, dtype=np.float32)
    site_p = np.stack(site_p).astype(np.float32) if site_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    hour_to_i, hour_n, hour_p = {}, [], []
    for h in hour_keys:
        mask = prior_df['hour_utc'].astype(int).values == h
        hour_to_i[h] = len(hour_n)
        hour_n.append(mask.sum())
        hour_p.append(Y_prior[mask].mean(axis=0))
    hour_n = np.array(hour_n, dtype=np.float32)
    hour_p = np.stack(hour_p).astype(np.float32) if hour_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), np.float32)

    return dict(
        global_p=global_p,
        site_to_i=site_to_i, site_n=site_n, site_p=site_p,
        hour_to_i=hour_to_i, hour_n=hour_n, hour_p=hour_p,
        sh_to_i=sh_to_i,     sh_n=sh_n,     sh_p=sh_p,
    )


def prior_logits(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables['global_p'][None, :], n, axis=0).astype(np.float32, copy=True)

    si  = np.fromiter((tables['site_to_i'].get(str(s), -1) for s in sites), np.int32, n)
    hi  = np.fromiter(
        (tables['hour_to_i'].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        np.int32, n
    )
    shi = np.fromiter(
        (tables['sh_to_i'].get((str(s), int(h)), -1) if int(h) >= 0 else -1
         for s, h in zip(sites, hours)),
        np.int32, n
    )

    valid = hi >= 0
    if valid.any():
        nh = tables['hour_n'][hi[valid]][:, None]
        p[valid] = nh / (nh + 8.0) * tables['hour_p'][hi[valid]] + (1.0 - nh / (nh + 8.0)) * p[valid]

    valid = si >= 0
    if valid.any():
        ns = tables['site_n'][si[valid]][:, None]
        p[valid] = ns / (ns + 8.0) * tables['site_p'][si[valid]] + (1.0 - ns / (ns + 8.0)) * p[valid]

    valid = shi >= 0
    if valid.any():
        nsh = tables['sh_n'][shi[valid]][:, None]
        p[valid] = nsh / (nsh + 4.0) * tables['sh_p'][shi[valid]] + (1.0 - nsh / (nsh + 4.0)) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)

def smooth_cols(scores, cols, alpha=0.35):
    """Weighted kernel smoothing for texture species (frogs, insects).
    This applies a center-weighted window to preserve the current signal
    while gracefully attenuating neighbor noise."""
    if len(cols) == 0: return scores.copy()
    # Scalar fallback to avoid broadcast mismatch with 5-window kernel
    a = alpha.max() if isinstance(alpha, np.ndarray) else alpha
    s = scores.copy(); view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    # Center-weighted 5-element weighted kernel: [0.1, 0.2, 0.4, 0.2, 0.1]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    prev2 = np.concatenate([x[:, :1, :], prev[:, :-1, :]], axis=1)
    nxt2  = np.concatenate([nxt[:, 1:, :], x[:, -1:, :]], axis=1)
    if isinstance(alpha, np.ndarray) and len(alpha) == 5:
        view[:, :, cols] = (alpha[0] * prev2 + alpha[1] * prev + alpha[2] * x + alpha[3] * nxt + alpha[4] * nxt2)
    else:
    view[:, :, cols] = (1.0 - a) * x + a * (0.1 * prev2 + 0.2 * prev + 0.4 * x + 0.2 * nxt + 0.1 * nxt2)
    return s


def smooth_events(scores, cols, alpha=0.15):
    """Local-max smoothing for event species (birds) -- preserves transient calls."""
    if len(cols) == 0: return scores.copy()
    # Scalar fallback to avoid broadcast mismatch with 5-window kernel
    a = alpha.max() if isinstance(alpha, np.ndarray) else alpha
    s = scores.copy(); view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - a) * x + a * np.maximum(x, np.maximum(prev, nxt))
    return s


def fuse_scores(base, sites, hours, tables):
    """Combine raw Perch logits with Bayesian prior offsets."""
    scores = base.copy()
    prior  = prior_logits(sites, hours, tables)
    if len(idx_mapped_active_event):   scores[:, idx_mapped_active_event]   += CFG.LAMBDA_EVENT * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture): scores[:, idx_mapped_active_texture] += CFG.LAMBDA_TEXTURE * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture): scores[:, idx_selected_proxy_active_texture] += CFG.LAMBDA_PROXY_TEXTURE * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):   scores[:, idx_selected_prioronly_active_event]   = CFG.LAMBDA_EVENT * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture): scores[:, idx_selected_prioronly_active_texture] = CFG.LAMBDA_TEXTURE * prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive): scores[:, idx_unmapped_inactive] = -8.0
    scores = smooth_cols(scores, idx_active_texture, alpha=CFG.SMOOTH_TEXTURE)
    scores = smooth_events(scores, idx_active_event, alpha=CFG.SMOOTH_EVENT)
    return scores.astype(np.float32), prior


## 8. Out-of-Fold Meta-Features

Probe input features must be produced out-of-fold to prevent data leakage. Prior tables for each validation fold are fitted on training folds only, with groups defined by filename to prevent any file from spanning both sets.


In [ ]:
def build_oof_base_prior(scores_raw, meta_df, sc_clean, Y_SC, n_splits=CFG.N_SPLITS):
    """Build honest out-of-fold fused scores and prior logits."""
    groups = meta_df["filename"].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)
    oof_base  = np.zeros_like(scores_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_raw, dtype=np.float32)
    fold_id   = np.full(len(meta_df), -1, dtype=np.int16)
    splits = list(gkf.split(scores_raw, groups=groups))
    for fold, (tr_idx, va_idx) in enumerate(tqdm(splits, desc="OOF folds"), 1):
        val_files = set(meta_df.iloc[va_idx]["filename"].tolist())
        prior_mask = ~sc_clean["filename"].isin(val_files).values
        tables = fit_prior_tables(sc_clean.loc[prior_mask].reset_index(drop=True), Y_SC[prior_mask])
        va_base, va_prior = fuse_scores(
            scores_raw[va_idx], sites=meta_df.iloc[va_idx]["site"].to_numpy(),
            hours=meta_df.iloc[va_idx]["hour_utc"].to_numpy(), tables=tables)
        oof_base[va_idx] = va_base; oof_prior[va_idx] = va_prior; fold_id[va_idx] = fold
    assert (fold_id >= 0).all()
    return oof_base, oof_prior, fold_id


def macro_auc_skip_empty(y_true, y_score):
    """Macro-averaged ROC-AUC, skipping classes with no positive labels."""
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")


OOF_CACHE = CFG.WORK_CACHE / "oof_meta.npz"
if OOF_CACHE.exists():
    print(f"Loading cached OOF from: {OOF_CACHE}")
    arr = np.load(OOF_CACHE)
    oof_base = arr["oof_base"].astype(np.float32)
    oof_prior = arr["oof_prior"].astype(np.float32)
    oof_fold_id = arr["fold_id"].astype(np.int16)
else:
    print("Building OOF meta-features...")
    oof_base, oof_prior, oof_fold_id = build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC)
    np.savez_compressed(OOF_CACHE, oof_base=oof_base, oof_prior=oof_prior, fold_id=oof_fold_id)
    print(f"Saved OOF to: {OOF_CACHE}")

baseline_oof_auc = macro_auc_skip_empty(Y_FULL, oof_base)
print(f"OOF baseline AUC: {baseline_oof_auc:.6f}")


## 9. Embedding Probes

Per-class Logistic Regression probes are trained on PCA-compressed embeddings augmented with temporal context features (previous/next window scores, file-level mean/max/std). The probes learn species-specific decision boundaries that complement the Perch logits. Interaction terms (raw x prior, raw x base) capture joint signal strength.


In [ ]:
def seq_features_1d(v):
    """Temporal context: neighbor values plus file-level statistics."""
    x = v.reshape(-1, CFG.N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    return prev_v, next_v, np.repeat(x.mean(axis=1), CFG.N_WINDOWS), np.repeat(x.max(axis=1), CFG.N_WINDOWS), np.repeat(x.std(axis=1), CFG.N_WINDOWS)


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    """Assemble feature vector for a single species classifier."""
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)
    diff_mean = base_col - mean_base
    diff_prev = base_col - prev_base
    diff_next = base_col - next_base
    feats = np.concatenate([
        emb_proj, raw_col[:, None], prior_col[:, None], base_col[:, None],
        prev_base[:, None], next_base[:, None], mean_base[:, None], max_base[:, None], std_base[:, None],
        diff_mean[:, None], diff_prev[:, None], diff_next[:, None],
        (raw_col * prior_col)[:, None], (raw_col * base_col)[:, None], (prior_col * base_col)[:, None],
    ], axis=1)
    return feats.astype(np.float32, copy=False)


In [ ]:
# PCA on all trusted embeddings
emb_scaler = StandardScaler()
emb_full_scaled = emb_scaler.fit_transform(emb_full)
n_comp = min(CFG.PCA_DIM, emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)
print(f"Embedding PCA: {emb_full.shape[1]}d -> {Z_FULL.shape[1]}d  (explained variance: {emb_pca.explained_variance_ratio_.sum():.4f})")

# train per-class Logistic Regression probes with positive oversampling
full_pos_counts = Y_FULL.sum(axis=0)
PROBE_CLASS_IDX = np.where(full_pos_counts >= CFG.MIN_POS)[0].astype(np.int32)
probe_models = {}

for cls_idx in tqdm(PROBE_CLASS_IDX, desc="Training probes"):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y): continue
    X_cls = build_class_features(Z_FULL, scores_full_raw[:, cls_idx], oof_prior[:, cls_idx], oof_base[:, cls_idx])
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    clf = LogisticRegression(penalty='l2', C=0.5, class_weight='balanced', max_iter=1000, random_state=42)
    clf.fit(X_cls, y)
    probe_models[cls_idx] = clf

print(f"Trained probes: {len(probe_models)} species")


## 10. Test Inference and Submission

Hidden test soundscapes are loaded, Perch inference is run, prior fusion is applied, and per-class probe predictions are blended with fused base scores. Final probabilities are written to submission.csv.


In [ ]:
test_paths = sorted((CFG.BASE / "test_soundscapes").glob("*.ogg"))
if len(test_paths) == 0:
    print(f"Hidden test not mounted. Dry-run on first {CFG.DRYRUN_FILES} train soundscapes.")
    test_paths = sorted((CFG.BASE / "train_soundscapes").glob("*.ogg"))[:CFG.DRYRUN_FILES]
else:
    print(f"Hidden test files: {len(test_paths)}")

meta_test, scores_test_raw, emb_test = infer_perch_batch(test_paths)
print(f"Test windows: {meta_test.shape[0]}")


In [ ]:
# fuse test scores with priors built on all labeled soundscapes
final_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)
scores_test_fused, prior_test = fuse_scores(
    scores_test_raw, sites=meta_test["site"].to_numpy(),
    hours=meta_test["hour_utc"].to_numpy(), tables=final_tables)

# apply probe predictions
emb_test_scaled = emb_scaler.transform(emb_test)
Z_test = emb_pca.transform(emb_test_scaled).astype(np.float32)
scores_final = scores_test_fused.copy()

for cls_idx, clf in probe_models.items():
    X_test_cls = build_class_features(
        Z_test, scores_test_raw[:, cls_idx], prior_test[:, cls_idx], scores_test_fused[:, cls_idx])
    pred_prob = clf.predict_proba(X_test_cls)[:, 1].astype(np.float32)
    pred_logit = np.log(pred_prob + 1e-7) - np.log(1 - pred_prob + 1e-7)
    scores_final[:, cls_idx] = (1.0 - CFG.PROBE_ALPHA) * scores_test_fused[:, cls_idx] + CFG.PROBE_ALPHA * pred_logit

print(f"Probes applied to {len(probe_models)} species columns.")


In [ ]:
# --- Inference Post-processing Architecture ---
# 1. File-level confidence scaling:
#    Multiplies chunk predictions by the per-file maximum for each species
#    to amplify dense acoustic events and natively suppress sparse noise.
# 2. Probability sharpening:
#    Applies a non-linear power activation to widen the delta between high
#    confidence foreground calls and low confidence background clutter.

# convert logits to probabilities
probs = 1.0 / (1.0 + np.exp(np.clip(-scores_final, -50, 50)))

# file-level confidence scaling
n_test_files = probs.shape[0] // CFG.N_WINDOWS
probs_3d = probs.reshape(n_test_files, CFG.N_WINDOWS, N_CLASSES)
file_max = probs_3d.max(axis=1, keepdims=True)
probs_3d = probs_3d * file_max
probs = probs_3d.reshape(-1, N_CLASSES)

# power sharpening
POST_POWER = 1.66
probs = np.power(np.clip(probs, 0, 1), POST_POWER)

submission = pd.DataFrame(probs, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)
submission = submission[sample_sub.columns.tolist()]

try:
    import kaggle_toolbox as tb
    tb.check_submission(submission, sample_sub)
except ImportError:
    pass
submission.to_csv("submission.csv", index=False)

print(f"Submission shape: {submission.shape}")
print(f"Post-processing: file-level scaling + sharpening power {POST_POWER}")
print("Saved to: submission.csv")
submission.head(3)


## 11. Analysis Summary

This notebook implements a multi-stage pipeline for acoustic species identification in the Pantanal:

1. **Perch v2** extracts 1536-d embeddings and 234-class logits from 5-second audio windows.
2. **Bayesian prior fusion** incorporates site and time-of-day prevalence with class-type-specific weights, distinguishing transient bird calls from continuous frog and insect choruses.
3. **Logistic Regression probes** on PCA-compressed embeddings with temporal context features learn per-species decision boundaries that complement the Perch logits.
4. **Temporal smoothing** uses weighted-kernel averaging for texture species and local-max pooling for event species.
5. **File-level confidence scaling** applies an adaptive multiplier using per-file species maxima, structurally boosting consistently detected foreground species.
6. **Probability sharpening** applies a non-linear exponent to widen the probabilistic margin between true acoustic events and background clutter.

The final submission blends fused Perch logits with probe predictions using a tuned mixing weight (alpha=0.45), followed by post-processing calibration.

***

**Citation:** Stefan Kahl, Tom Denton, Larissa Sugai, Liliana Piatti, Ryan Holbrook, Holger Klinck, and Ashley Oldacre. BirdCLEF+ 2026. https://kaggle.com/competitions/birdclef-2026, 2026. Kaggle.
